In [ ]:
# ==========================================
# 1: SETUP & MODEL LOADING
# ==========================================
!pip install open_clip_torch peft > /dev/null

import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler # AMP added for speed
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                             roc_curve, recall_score, f1_score, balanced_accuracy_score)
import matplotlib.pyplot as plt
import seaborn as sns
import open_clip
from PIL import Image
from peft import LoraConfig, get_peft_model, PeftModel

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Paths
base_path = "/content/drive/MyDrive/MrNet-v1/MRNet-v1.0"
axial_dir = base_path + "/merged_data/axial"
coronal_dir  = base_path + "/merged_data/coronal"
sagittal_dir = base_path + "/merged_data/sagittal"

train_df = pd.read_csv(base_path + "/train_split.csv")
val_df   = pd.read_csv(base_path + "/val_split.csv")
test_df  = pd.read_csv(base_path + "/test_split.csv")

# Load Base BioMedCLIP
print("Loading Base BioMedCLIP...")
model_name = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
clip_model, _, preprocess = open_clip.create_model_and_transforms(model_name)
clip_model = clip_model.to(device)
tokenizer = open_clip.get_tokenizer(model_name)
print("Loaded successfully!")

Mounted at /content/drive
Using device: cuda
Loading Base BioMedCLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Loaded successfully!


In [ ]:
# ==========================================
#  2: DATASET & AUGMENTATIONS
# ==========================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
])

class LoRAMRNetDataset(Dataset):
    def __init__(self, df, plane_dir, is_train=False):
        self.df = df
        self.plane_dir = plane_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patient_id = str(self.df.iloc[idx]["id"]).zfill(4)
        label = self.df.iloc[idx]["label"]

        vol = np.load(os.path.join(self.plane_dir, patient_id + ".npy"))
        vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)

        processed_slices = []
        for i in range(vol.shape[0]):
            slice_img = cv2.resize(vol[i], (224, 224))
            slice_img = (slice_img * 255).astype(np.uint8)
            slice_rgb = np.stack((slice_img,)*3, axis=-1)

            if self.is_train:
                slice_rgb = np.array(train_transform(slice_rgb))

            img_pil = Image.fromarray(slice_rgb)
            img_tensor = preprocess(img_pil)
            processed_slices.append(img_tensor)

        return torch.stack(processed_slices), torch.tensor(label, dtype=torch.long), patient_id

# Sampler for Class Imbalance
labels = train_df["label"].values
class_counts = np.bincount(labels)
sample_weights = (1. / class_counts)[labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

In [ ]:
# ==========================================
#  3: GATED ATTENTION, LoRA & TEXT FEATURES
# ==========================================
def get_ensembled_text_features(plane_name):
    prompts_healthy = [
        f"A normal {plane_name} MRI slice of the knee with an intact anterior cruciate ligament.",
        f"Healthy knee MRI showing a continuous ACL."
    ]
    prompts_tear = [
        f"A {plane_name} MRI slice of the knee showing a complete tear of the anterior cruciate ligament.",
        f"Ruptured anterior cruciate ligament with discontinuity."
    ]

    clip_model.eval()
    with torch.no_grad():
        tok_healthy = tokenizer(prompts_healthy).to(device)
        feat_healthy = clip_model.encode_text(tok_healthy).mean(dim=0, keepdim=True)

        tok_tear = tokenizer(prompts_tear).to(device)
        feat_tear = clip_model.encode_text(tok_tear).mean(dim=0, keepdim=True)

        text_features = torch.cat([feat_healthy, feat_tear], dim=0)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    return text_features

# NEW: Gated Attention Network (Replaces Mean Pooling)
class GatedAttention(nn.Module):
    def __init__(self, embed_dim=512, hidden_dim=256):
        super().__init__()
        self.attention_V = nn.Linear(embed_dim, hidden_dim)
        self.attention_U = nn.Linear(embed_dim, hidden_dim)
        self.attention_weights = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x shape: [num_slices, embed_dim]
        a_v = torch.tanh(self.attention_V(x))
        a_u = torch.sigmoid(self.attention_U(x))
        w = self.attention_weights(a_v * a_u)       # [num_slices, 1]
        A = torch.softmax(w, dim=0)                 # [num_slices, 1]

        # Weighted sum of slices based on attention score
        M = torch.sum(A * x, dim=0, keepdim=True)   # [1, embed_dim]
        return M, A

class LoRABioMedCLIP(nn.Module):
    def __init__(self, lora_vision_encoder, base_clip_model, text_features):
        super().__init__()
        self.vision_encoder = lora_vision_encoder
        self.text_features = text_features.detach()
        self.logit_scale = base_clip_model.logit_scale

        # BioMedCLIP's embedding dimension is 512
        self.attention = GatedAttention(embed_dim=512, hidden_dim=256)

    def forward(self, image_volume):
        image_features = self.vision_encoder(image_volume)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Apply Gated Attention instead of Mean()
        patient_image_feature, attn_weights = self.attention(image_features)
        patient_image_feature = patient_image_feature / patient_image_feature.norm(dim=-1, keepdim=True)

        logit_scale = self.logit_scale.exp()
        logits = logit_scale * patient_image_feature @ self.text_features.t()
        return logits, attn_weights

In [ ]:
# ==========================================
# 4: LoRA + ATTENTION TRAINING LOOP (with AMP)
# ==========================================
def train_lora_model(planes, epochs=10):
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler() # AMP for faster training

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["fc1", "fc2", "c_fc", "c_proj", "out_proj", "proj"],
        lora_dropout=0.1,
        bias="none"
    )

    for plane_name, plane_dir in planes:
        print(f"\n{'='*50}\n STARTING PEFT LoRA + GATED ATTN TUNING ({plane_name.upper()})\n{'='*50}")

        train_ds = LoRAMRNetDataset(train_df, plane_dir, is_train=True)
        val_ds   = LoRAMRNetDataset(val_df, plane_dir, is_train=False)

        train_loader = DataLoader(train_ds, batch_size=1, sampler=sampler, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

        text_feats = get_ensembled_text_features(plane_name)

        lora_vision_encoder = get_peft_model(clip_model.visual, lora_config)

        model = LoRABioMedCLIP(lora_vision_encoder, clip_model, text_feats).to(device)

        # Optimize BOTH LoRA adapters AND Gated Attention parameters
        trainable_params = list(filter(lambda p: p.requires_grad, model.vision_encoder.parameters())) + \
                           list(model.attention.parameters())

        optimizer = optim.AdamW(trainable_params, lr=2e-4, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

        best_auc = 0.0

        for epoch in range(epochs):
            model.train()
            t_loss, t_correct, t_total = 0, 0, 0

            for imgs, lbls, _ in train_loader:
                imgs, lbls = imgs.squeeze(0).to(device), lbls.to(device)
                optimizer.zero_grad()

                # AMP Context Manager for speed
                with autocast('cuda'):
                    logits, _ = model(imgs)
                    loss = criterion(logits, lbls)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                t_loss += loss.item()
                preds = logits.argmax(dim=1)
                t_correct += (preds == lbls).sum().item()
                t_total += 1

            scheduler.step()

            model.eval()
            v_loss, v_correct, v_total, y_true, y_probs = 0, 0, 0, [], []
            with torch.no_grad():
                for imgs, lbls, _ in val_loader:
                    imgs, lbls = imgs.squeeze(0).to(device), lbls.to(device)

                    with autocast('cuda'):
                        logits, _ = model(imgs)
                        v_loss += criterion(logits, lbls).item()

                    prob_tear = logits.softmax(dim=-1)[0, 1].item()
                    y_true.append(lbls.item())
                    y_probs.append(prob_tear)

                    if logits.argmax(dim=1).item() == lbls.item(): v_correct += 1
                    v_total += 1

            auc = roc_auc_score(y_true, y_probs)

            print(f"Epoch [{epoch+1}/{epochs}] | LR: {scheduler.get_last_lr()[0]:.2e}")
            print(f"  [Train] Loss: {t_loss/t_total:.4f} | Acc: {(t_correct/t_total)*100:.2f}%")
            print(f"  [Val]   Loss: {v_loss/v_total:.4f} | Acc: {(v_correct/v_total)*100:.2f}% | AUC: {auc:.4f}")
            print("-" * 45)

            if auc > best_auc:
                best_auc = auc
                # Save both LoRA weights AND Attention weights
                model.vision_encoder.save_pretrained(os.path.join(base_path, f"lora_{plane_name}_best"))
                torch.save(model.attention.state_dict(), os.path.join(base_path, f"attn_{plane_name}_best.pth"))
                print("  => New Best Models Saved!")

        clip_model.visual = lora_vision_encoder.unload()

In [ ]:
target_planes = [("coronal", coronal_dir)]
train_lora_model(target_planes, epochs=10)


 STARTING PEFT LoRA + GATED ATTN TUNING (CORONAL)
Epoch [1/10] | LR: 1.95e-04
  [Train] Loss: 0.6139 | Acc: 68.91%
  [Val]   Loss: 0.4854 | Acc: 80.00% | AUC: 0.8186
---------------------------------------------
  => New Best Models Saved!
Epoch [2/10] | LR: 1.81e-04
  [Train] Loss: 0.4685 | Acc: 78.40%
  [Val]   Loss: 0.7792 | Acc: 66.40% | AUC: 0.8543
---------------------------------------------
  => New Best Models Saved!
Epoch [3/10] | LR: 1.59e-04
  [Train] Loss: 0.3359 | Acc: 84.34%
  [Val]   Loss: 0.3727 | Acc: 87.20% | AUC: 0.8932
---------------------------------------------
  => New Best Models Saved!
Epoch [4/10] | LR: 1.31e-04
  [Train] Loss: 0.2666 | Acc: 87.09%
  [Val]   Loss: 0.4622 | Acc: 85.60% | AUC: 0.8695
---------------------------------------------
Epoch [5/10] | LR: 1.01e-04
  [Train] Loss: 0.2026 | Acc: 92.46%
  [Val]   Loss: 0.3713 | Acc: 88.00% | AUC: 0.9118
---------------------------------------------
  => New Best Models Saved!
Epoch [6/10] | LR: 6.98e-05